# Pediatric Bone Age Predictor — Google Colab / Kaggle Training

**Dataset**: [RSNA Pediatric Bone Age](https://www.kaggle.com/competitions/rsna-bone-age)

**GPU Setup**: `Runtime` → `Change runtime type` → `T4 GPU` (or A100 / V100)

This pipeline trains and evaluates:
1. **CNN Baseline** (ResNet-18/50 backbone + transfer learning with preserved ImageNet filters)
2. **CNN + DNN** (Dense head with LayerNorm/Dropout)
3. **Multimodal CNN** (Image CNN branch fused with embedded gender metadata)
4. **CNN + Random Forest** (CNN features + Random Forest regressor)

In [ ]:
# Step 1: Clone repository or navigate to backend
!git clone https://github.com/Nidhishri2005/Bonetest.git
%cd Bonetest/backend
!pip install -r requirements.txt

In [ ]:
# Step 2: Download or set RSNA dataset directory
# Option A: Upload Kaggle API key (kaggle.json) and download directly
# !pip install kaggle
# !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c rsna-bone-age -p /content/rsna
# !unzip -q /content/rsna/rsna-bone-age.zip -d /content/rsna

DATA_DIR = '/content/rsna'  # Update with your RSNA dataset path
import os
assert os.path.exists(DATA_DIR), f'Dataset path not found: {DATA_DIR}'

In [ ]:
# Step 3: Train Models with Target Standardization & Two-Phase Transfer Learning

# 1. Train CNN Baseline (ResNet-18)
!python -m ml.train --model-type cnn --backbone resnet18 --data-dir {DATA_DIR} --epochs 30 --batch-size 16 --lr 1e-4 --loss smooth_l1 --pretrained

# 2. Train CNN + DNN
!python -m ml.train --model-type cnn_dnn --backbone resnet18 --data-dir {DATA_DIR} --epochs 30 --batch-size 16 --lr 1e-4 --loss smooth_l1 --pretrained

# 3. Train Multimodal CNN (Image + Gender metadata)
!python -m ml.train --model-type multimodal_cnn --backbone resnet18 --data-dir {DATA_DIR} --epochs 30 --batch-size 16 --lr 1e-4 --loss smooth_l1 --pretrained

# 4. Train CNN + Random Forest
!python -m ml.train --model-type cnn_rf --backbone resnet18 --data-dir {DATA_DIR} --batch-size 32

In [ ]:
# Step 4: Evaluate Models on Validation Set & Generate Authentic Reports
!python -m ml.evaluate --data-dir {DATA_DIR} --checkpoints-dir checkpoints --metrics-dir metrics
!python -m ml.compare_models

# Display comparison table
import pandas as pd
report = pd.read_csv('metrics/comparison_report.csv')
print(report.to_markdown(index=False))

In [ ]:
# Step 5: Archive artifacts for backend deployment
!zip -r bone_age_artifacts.zip checkpoints rf_models metrics
print('Download bone_age_artifacts.zip and extract into backend/ for deployment.')